<a href="https://colab.research.google.com/github/mohiuddinshahrukh/PM_STEDE_25-26/blob/main/ASPECT_DATA__SET_ARG_MIN_25_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Cell 1: Install & Setup
import os

# 1. Install Libraries
!pip install amrlib spacy transformers torch pandas --quiet
!python -m spacy download en_core_web_sm --quiet

# 2. Download the AMR Parsing Model (T5-based)
# We manually download it because it's not included in the pip install
model_url = "https://github.com/bjascob/amrlib-models/releases/download/model_parse_t5-v0_2_0/model_parse_t5-v0_2_0.tar.gz"
model_tar = "model_parse_t5-v0_2_0.tar.gz"
model_dir = "model_parse_t5-v0_2_0"

if not os.path.isdir(model_dir):
    print("⬇️ Downloading AMR Model (approx 800MB)...")
    !wget -q {model_url}
    print("📦 Extracting Model...")
    !tar -xzf {model_tar}
    print("✅ Model Ready!")
else:
    print("✅ Model already downloaded.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 147.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
⬇️ Downloading AMR Model (approx 800MB)...
📦 Extracting Model...
✅ Model Ready!


In [2]:
!pip install unidecode


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 8.9 MB/s eta 0:00:00


In [4]:
# Cell 2: Run AMR Parsing
import amrlib
import pandas as pd
import torch

# 1. Load the Model we just downloaded
# We point 'model_dir' to the folder we extracted in Step 1
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Loading model on {device}...")

stog = amrlib.load_stog_model(model_dir='model_parse_t5-v0_2_0', device=device)

# 2. Create Sample Data (Since the official CSV is restricted)
# These are real examples similar to the UKP dataset
data = {
    'sentence1': [
        "Fracking can contaminate water and water wells.",
        "The death penalty is a legal punishment.",
        "School uniforms reduce peer pressure."
    ],
    'sentence2': [
        "Hydraulic fracturing poses risks to groundwater quality.",
        "Capital punishment should be abolished.",
        "Uniforms take away student individuality."
    ],
    'topic': ["Fracking", "Death Penalty", "School Uniforms"]
}
# --- THE FIX ---
# Instead of converting to CSV and reading it back, just make the DataFrame directly:
df = pd.DataFrame(data)

# 3. Parse the Arguments
print(f"\nParsing {len(df)} argument pairs...\n")

# We zip the two columns to parse them together
args1 = df['sentence1'].tolist()
args2 = df['sentence2'].tolist()

# Parse all sentences at once
graphs1 = stog.parse_sents(args1)
graphs2 = stog.parse_sents(args2)

# 4. Display Results
for i in range(len(df)):
    print(f"Topic: {df['topic'][i]}")
    print(f"Argument 1: {args1[i]}")
    print(f"AMR Graph 1:\n{graphs1[i]}")
    print("-" * 20)
    print(f"Argument 2: {args2[i]}")
    print(f"AMR Graph 2:\n{graphs2[i]}")
    print("=" * 50 + "\n")

🚀 Loading model on cuda...

Parsing 3 argument pairs...

Topic: Fracking
Argument 1: Fracking can contaminate water and water wells.
AMR Graph 1:
# ::snt Fracking can contaminate water and water wells.
(p / possible-01
      :ARG1 (c / contaminate-01
            :ARG0 (f / fracking)
            :ARG1 (a / and
                  :op1 (w / water)
                  :op2 (w2 / well
                        :mod w))))
--------------------
Argument 2: Hydraulic fracturing poses risks to groundwater quality.
AMR Graph 2:
# ::snt Hydraulic fracturing poses risks to groundwater quality.
(p / pose-02
      :ARG0 (f / fracture-01
            :mod (h / hydraulic))
      :ARG1 (r / risk-01
            :ARG2 (q / quality
                  :mod (g / groundwater))))

Topic: Death Penalty
Argument 1: The death penalty is a legal punishment.
AMR Graph 1:
# ::snt The death penalty is a legal punishment.
(p / punish-01
      :ARG2 (d / die-01)
      :ARG1-of (l / legal-02))
--------------------
Argument 2: 

In [5]:
# Cell 3: Calculate Similarity (Smatch & Concept Overlap)

# 1. Install smatch (Standard AMR metric library)
!pip install smatch --quiet
import smatch
import re

def get_concepts(amr_string):
    """
    Extracts concepts (e.g., 'contaminate-01', 'fracking') from the graph.
    This mimics the 'Concept-focused' metric from Opitz et al..
    """
    # Regex to find (variable / concept) patterns
    # We grab the 'concept' part
    concepts = re.findall(r'\/ ([a-zA-Z0-9\-]+)', amr_string)
    return set(concepts)

def calculate_similarity(graph1, graph2):
    # 1. Structural Similarity (Smatch)
    # get_amr_match returns (f-score, precision, recall)
    # We pass the generator of the single graph
    structural_score = next(smatch.score_amr_pairs([graph1], [graph2]))[2] # Index 2 is F-score

    # 2. Concept Similarity (Jaccard Index)
    # A simple approximation of the paper's 'Concept-Focus' metric
    c1 = get_concepts(graph1)
    c2 = get_concepts(graph2)

    if not c1 or not c2: return 0.0, 0.0, c1, c2

    intersection = c1.intersection(c2)
    union = c1.union(c2)
    concept_score = len(intersection) / len(union)

    return structural_score, concept_score, intersection, c1, c2

# --- Run Analysis on your Data ---
print(f"{'Topic':<15} | {'Struct. Score':<12} | {'Concept Score':<12} | {'Shared Concepts'}")
print("-" * 80)

for i in range(len(df)):
    g1 = graphs1[i]
    g2 = graphs2[i]
    topic = df['topic'][i]

    s_score, c_score, shared, c1, c2 = calculate_similarity(g1, g2)

    # Formatting for display
    shared_str = ", ".join(list(shared)) if shared else "None"

    print(f"{topic:<15} | {s_score:.3f}        | {c_score:.3f}        | {shared_str}")

Topic           | Struct. Score | Concept Score | Shared Concepts
--------------------------------------------------------------------------------
Fracking        | 0.000        | 0.000        | None
Death Penalty   | 0.000        | 0.167        | punish-01
School Uniforms | 0.000        | 0.111        | uniform
